# 🚔 CivicCam YOLOv8 Training Notebook

This notebook trains a YOLOv8 model for littering detection.

**Classes:**
- `0`: license_plate
- `1`: object
- `2`: public (people)
- `3`: waste (litter)

---

## Step 1: Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install YOLOv8 (Ultralytics)
!pip install ultralytics -q

## Step 2: Upload Your Dataset

### Option A: Upload ZIP directly (smaller datasets)
Run the cell below, click "Choose Files", and select your dataset `.zip` file.

In [ ]:
# OPTION A: Direct upload from your computer
from google.colab import files
import os

print("📁 Click 'Choose Files' to upload your dataset.zip")
uploaded = files.upload()

# Get the uploaded filename
zip_filename = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {zip_filename}")

In [ ]:
# Unzip the dataset
!unzip -q "{zip_filename}" -d /content/dataset
!ls /content/dataset

### Option B: Load from Google Drive (larger datasets)
If your dataset is large (>100MB), upload the `.zip` to Google Drive first, then run below.

In [ ]:
# OPTION B: Mount Google Drive and copy from there
from google.colab import drive
drive.mount('/content/drive')

# ⚠️ EDIT THIS PATH to match your zip file location in Drive
DRIVE_ZIP_PATH = "/content/drive/MyDrive/civiccam_dataset.zip"

!unzip -q "{DRIVE_ZIP_PATH}" -d /content/dataset
!ls /content/dataset

## Step 3: Verify Dataset Structure

Your dataset folder should look like:
```
dataset/
├── data.yaml
├── train/
│   ├── images/
│   └── labels/
└── valid/
    ├── images/
    └── labels/
```

In [ ]:
# Find the data.yaml file
import os

data_yaml_path = None
for root, dirs, files_list in os.walk('/content/dataset'):
    if 'data.yaml' in files_list:
        data_yaml_path = os.path.join(root, 'data.yaml')
        break

if data_yaml_path:
    print(f"✅ Found data.yaml at: {data_yaml_path}")
    print("\n--- Contents ---")
    !cat "{data_yaml_path}"
else:
    print("❌ data.yaml not found! Check your dataset structure.")

In [ ]:
# Fix data.yaml paths (required for Colab)
import yaml

with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

# Update the path to point to dataset folder
dataset_root = os.path.dirname(data_yaml_path)
data_config['path'] = dataset_root

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print("✅ Updated data.yaml paths:")
!cat "{data_yaml_path}"

## Step 4: Train the Model 🚀

In [ ]:
from ultralytics import YOLO

# Load the base YOLOv8 model (nano version for faster training)
# Options: yolov8n.pt (nano), yolov8s.pt (small), yolov8m.pt (medium)
model = YOLO('yolov8n.pt')

# Start training
results = model.train(
    data=data_yaml_path,
    epochs=100,           # Increase for better accuracy (50-150 typical)
    imgsz=640,            # Image size
    batch=16,             # Reduce to 8 if you get OOM errors
    name='civiccam',
    patience=20,          # Early stopping patience
    save=True,
    plots=True
)

## Step 5: View Training Results

In [ ]:
# Display training results
from IPython.display import Image, display
import glob

# Find the latest run folder
run_folders = sorted(glob.glob('/content/runs/detect/civiccam*'))
latest_run = run_folders[-1] if run_folders else None

if latest_run:
    print(f"📊 Training results from: {latest_run}")
    
    # Show confusion matrix
    if os.path.exists(f"{latest_run}/confusion_matrix.png"):
        display(Image(filename=f"{latest_run}/confusion_matrix.png", width=600))
    
    # Show results chart
    if os.path.exists(f"{latest_run}/results.png"):
        display(Image(filename=f"{latest_run}/results.png", width=800))

## Step 6: Download Trained Model

In [ ]:
# Download best.pt to your computer
from google.colab import files

best_model_path = f"{latest_run}/weights/best.pt"
print(f"📦 Downloading: {best_model_path}")
files.download(best_model_path)

In [ ]:
# OR save to Google Drive
!cp "{latest_run}/weights/best.pt" /content/drive/MyDrive/civiccam_best.pt
print("✅ Model saved to Google Drive as: civiccam_best.pt")

## Step 7: Test the Model (Optional)

In [ ]:
# Test on validation set
model = YOLO(f"{latest_run}/weights/best.pt")
metrics = model.val()
print(f"\n📈 mAP50: {metrics.box.map50:.3f}")
print(f"📈 mAP50-95: {metrics.box.map:.3f}")